In [ ]:
!pip -q install -U pip
!pip -q install --force-reinstall "numpy==2.0.2" "pandas==2.2.2"
!pip -q install --force-reinstall \
  "transformers==4.41.2" \
  "tokenizers==0.19.1" \
  "accelerate==0.30.1" \
  "huggingface-hub==0.23.4" \
  "safetensors>=0.4.3" \
  "bert-score==0.3.13"

!pip -q install pandas numpy tqdm bert-score
!pip -q install -U llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu121
!pip install rouge-score
!pip install --force-reinstall rouge-score
!pip install sacrebleu

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import torch
import psutil
from tqdm import tqdm

from bert_score import score as bert_score
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

ORI_JSON = "/content/drive/MyDrive/BioBERT_PubMedQA_Checkpoints/ori_pqal.json"
MODEL_PATH = "/content/drive/MyDrive/BioBERT_PubMedQA_Checkpoints/llama-2-7b-chat.Q4_0.gguf"

N_SAMPLES = 500
SEED = 42

OUT_JSON = "/content/drive/MyDrive/BioBERT_PubMedQA_Checkpoints/pqal_eval_nonrag.json"
OUT_METRICS_CSV = "/content/drive/MyDrive/BioBERT_PubMedQA_Checkpoints/pqal_nonrag_metrics.csv"

BERTSCORE_MODEL = "bert-base-uncased"
N_CTX = 2048
MAX_TOKENS = 120
TEMPERATURE = 0.2
TOP_P = 0.9
N_GPU_LAYERS = 35

np.random.seed(SEED)

def get_memory_usage():
    import psutil
    vm = psutil.virtual_memory()
    ram_used_mb = (vm.total - vm.available) / (1024**2)
    gpu_used_mb = 0.0
    try:
        import subprocess
        smi_out = subprocess.check_output(
            ['nvidia-smi', '--query-gpu=memory.used', '--format=csv,nounits,noheader'],
            encoding='utf-8'
        )
        gpu_used_mb = float(smi_out.strip().split('\n')[0])
    except Exception:
        if torch.cuda.is_available():
            gpu_used_mb = torch.cuda.memory_allocated() / (1024**2)
    return ram_used_mb, gpu_used_mb

def first_sentence(text):
    import re
    text = (text or "").strip()
    if not text:
        return text
    parts = re.split(r'(?<=[.!?])\s+', text)
    return parts[0].strip() if parts else text

with open(ORI_JSON, "r", encoding="utf-8") as f:
    data = json.load(f)

if isinstance(data, dict):
    records = [data[k] for k in sorted(data.keys(), key=lambda x: int(x) if x.isdigit() else x)]
else:
    records = data

print("Số mẫu:", len(records))
print("Key mẫu đầu:", records[0].keys())

for item in records:
    if "QUESTION" in item and "question" not in item:
        item["question"] = item["QUESTION"]
    if "LONG_ANSWER" in item and "long_answer" not in item:
        item["long_answer"] = item["LONG_ANSWER"]

df = pd.DataFrame(records)
print('Các cột df:', df.columns.tolist())

df = df[(df["question"].str.len() > 10) & (df["long_answer"].str.len() > 20)].copy()
df_sample = df.sample(n=min(N_SAMPLES, len(df)), random_state=SEED).reset_index(drop=True)
print(f"Sample size: {df_sample.shape}")

from llama_cpp import Llama
print("Loading Llama...")
llm = Llama(
    model_path=MODEL_PATH,
    n_ctx=N_CTX,
    n_threads=os.cpu_count(),
    n_gpu_layers=N_GPU_LAYERS,
    verbose=False,
)
ram_mb, gpu_mb = get_memory_usage()
print(f"Llama loaded. RAM: {ram_mb:.1f} MB | VRAM: {gpu_mb:.1f} MB")

def llama_generate_question_only(question):
    prompt = f"""[INST] You are a medical QA assistant.
Answer the following medical question as clearly as possible, in one conclusive sentence.

Question: {question}
[/INST]
"""
    out = llm(
        prompt,
        max_tokens=MAX_TOKENS,
        temperature=TEMPERATURE,
        top_p=TOP_P,
        stop=["</s>", "[INST]", "[/INST]"],
        echo=False
    )
    return first_sentence(out["choices"][0]["text"].strip())

results = []
gen_time_sum_ms = 0.0
gen_time_n_ok = 0
t_gen_all0 = time.time()

for idx, row in tqdm(df_sample.iterrows(), total=len(df_sample)):
    q = row["question"]
    gt_long = row["long_answer"]
    gt_label = row.get("final_decision", "")
    ram_before, gpu_before = get_memory_usage()
    try:
        t0 = time.time()
        gen = llama_generate_question_only(q)
        gen_ms = (time.time() - t0) * 1000.0
        gen_time_sum_ms += gen_ms
        gen_time_n_ok += 1
        ram_after, gpu_after = get_memory_usage()
        result = {
            "idx": idx,
            "question": q,
            "ground_truth_long": gt_long,
            "ground_truth_label": gt_label,
            "generated": gen,
            "gen_ms": float(gen_ms),
            "ram_before_mb": float(ram_before),
            "ram_after_mb": float(ram_after),
            "gpu_before_mb": float(gpu_before),
            "gpu_after_mb": float(gpu_after),
        }
        results.append(result)
    except Exception as e:
        results.append({
            "idx": idx,
            "question": q,
            "ground_truth_long": gt_long,
            "error": str(e)
        })

gen_elapsed_s = time.time() - t_gen_all0
print(f"Generation done: {len(results)} samples, elapsed: {gen_elapsed_s:.1f}s")
avg_ms = (gen_time_sum_ms / gen_time_n_ok) if gen_time_n_ok > 0 else float('nan')
throughput = len(results) / gen_elapsed_s if gen_elapsed_s > 0 else float('nan')
print(f"AVG gen time: {avg_ms:.2f} ms/item, Throughput: {throughput:.3f} samples/sec")

with open(OUT_JSON, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

cands, refs = [], []
rouge1s, rouge2s, rougels, bleus = [], [], [], []
for r in results:
    gen = r.get("generated", "")
    ref = r.get("ground_truth_long", "")
    if gen and ref:
        cands.append(gen)
        refs.append(ref)
        scorer = rouge_scorer.RougeScorer(['rouge1','rouge2','rougeL'], use_stemmer=True)
        scores = scorer.score(ref, gen)
        rouge1s.append(scores['rouge1'].fmeasure)
        rouge2s.append(scores['rouge2'].fmeasure)
        rougels.append(scores['rougeL'].fmeasure)
        bleu = sentence_bleu([ref.split()], gen.split(), smoothing_function=SmoothingFunction().method1)
        bleus.append(bleu)

ram_b, gpu_b = get_memory_usage()
t0 = time.time()
P, R, F1 = bert_score(
    cands, refs,
    lang="en",
    model_type=BERTSCORE_MODEL,
    verbose=False,
    rescale_with_baseline=True,
    device="cuda"
)
bs_ms = (time.time() - t0) * 1000.0
ram_a, gpu_a = get_memory_usage()
P = P.detach().cpu().numpy()
R = R.detach().cpu().numpy()
F1 = F1.detach().cpu().numpy()

n = len(cands)
metrics = {
    "n": n,
    "rouge1_mean": float(np.mean(rouge1s)),
    "rouge2_mean": float(np.mean(rouge2s)),
    "rougeL_mean": float(np.mean(rougels)),
    "bleu_mean": float(np.mean(bleus)),
    "bertscore_P_mean": float(P.mean()),
    "bertscore_R_mean": float(R.mean()),
    "bertscore_F1_mean": float(F1.mean()),
    "bertscore_F1_std": float(F1.std()),
    "avg_gen_ms": float(avg_ms),
    "total_gen_time_s": float(gen_elapsed_s),
    "throughput": float(throughput),
    "bertscore_total_ms": float(bs_ms),
    "bertscore_avg_ms_pair": float(bs_ms / max(n, 1)),
    "ram_before_bs_mb": ram_b,
    "gpu_before_bs_mb": gpu_b,
    "gpu_after_bs_mb": gpu_a
}
pd.DataFrame([metrics]).to_csv(OUT_METRICS_CSV, index=False)
print(metrics)

In [ ]:
import json
import numpy as np
import pandas as pd
from bert_score import score as bert_score
from rouge_score import rouge_scorer
import sacrebleu

OUT_JSON = "/content/drive/MyDrive/BioBERT_PubMedQA_Checkpoints/pqal_eval_nonrag.json"
OUT_METRICS_CSV = "/content/drive/MyDrive/BioBERT_PubMedQA_Checkpoints/pqal_nonrag_metrics.csv"

BERTSCORE_MODEL = "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext"

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

with open(OUT_JSON, "r", encoding="utf-8") as f:
    results = json.load(f)

cands, refs = [], []
for item in results:
    gen = item.get("generated", "")
    ref = item.get("ground_truth_long", "")
    if gen and ref:
        cands.append(gen)
        refs.append(ref)

n = len(cands)
if n == 0:
    raise ValueError("Không có dữ liệu hợp lệ để đánh giá!")

# --- 1. BERTScore ---
P, R, F1_bert = bert_score(
    cands, refs,
    model_type=BERTSCORE_MODEL,
    num_layers=12,
    lang="en",
    verbose=False,
    rescale_with_baseline=False,
    device="cuda"
)
P = P.detach().cpu().numpy()
R = R.detach().cpu().numpy()
F1_bert = F1_bert.detach().cpu().numpy()

# --- 2. ROUGE & BLEU ---
rouge1_scores, rouge2_scores, rougeL_scores, bleu_scores = [], [], [], []

for cand, ref in zip(cands, refs):
    r_scores = scorer.score(ref, cand)
    rouge1_scores.append(r_scores['rouge1'].fmeasure)
    rouge2_scores.append(r_scores['rouge2'].fmeasure)
    rougeL_scores.append(r_scores['rougeL'].fmeasure)
    bleu = sacrebleu.sentence_bleu(cand, [ref]).score / 100.0
    bleu_scores.append(bleu)

metrics = {
    "n": n,
    "bertscore_P_mean": float(P.mean()),
    "bertscore_R_mean": float(R.mean()),
    "bertscore_F1_mean": float(F1_bert.mean()),
    "bertscore_F1_std": float(F1_bert.std()),
    "rouge1_f1_mean": float(np.mean(rouge1_scores)),
    "rouge2_f1_mean": float(np.mean(rouge2_scores)),
    "rougeL_f1_mean": float(np.mean(rougeL_scores)),
    "bleu_mean": float(np.mean(bleu_scores)),
}

metrics_df = pd.DataFrame([metrics])
metrics_df.to_csv(OUT_METRICS_CSV, index=False)
print("Saved:", OUT_METRICS_CSV)
display(metrics_df)